# MyDigitalTwin — Analyse Fréquentielle des Centres d'Intérêt

**Objectif** : Identifier les concepts les plus fréquents dans mes données réelles (Spotify, YouTube, Google, Netflix, Chrome) pour calibrer les macro-catégories de la home page.

**Approche** : Analyse de fréquence de mots par source → identification des gaps dans `CATEGORY_KEYWORDS` → mise à jour du dictionnaire.

> **Pourquoi pas K-Means ?**  
> Une première tentative de clustering TF-IDF + K-Means a produit un cluster *catch-all* dominant (~73% des données, Silhouette ≈ 0.19). Le problème est structurel : les textes courts multi-sources (artistes Spotify, titres YouTube, requêtes Google) ont des espaces sémantiques trop hétérogènes pour être clusterisés conjointement.  
> L'analyse fréquentielle est plus directe et interprétable pour des catégories prédéfinies.

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("MyDigitalTwin-FrequencyAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# ── Config centrale ────────────────────────────────────────────────────────────
import sys as _sys, os as _os
_d = _os.path.abspath('')
while not _os.path.exists(_os.path.join(_d, 'config.py')):
    _p = _os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
if _d not in _sys.path: _sys.path.insert(0, _d)
from config import WAREHOUSE

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(table_name):
    return spark.read.format("delta").load(os.path.join(WAREHOUSE, table_name))


Warehouse: /opt/spark/data/warehouse


26/04/27 22:51:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## Étape 1 — Chargement des sources texte

On extrait la colonne texte pertinente de chaque source Delta, en gardant la provenance (`source`) pour analyser chaque canal séparément.

In [2]:
# ── 1. CHARGEMENT ─────────────────────────────────────────────────────────────
sources = {
    "Google Searches": read_table("google_searches").select(F.col("query").alias("text")),
    "YouTube":         read_table("youtube_watch").select(F.col("title").alias("text")),
    "Chrome":          read_table("google_chrome").select(F.col("title").alias("text")),
    "Spotify":         read_table("spotify_streams").select(F.col("artistName").alias("text")).dropDuplicates(["text"]),
    "Netflix":         read_table("netflix_views").select(F.col("show_title").alias("text")),
}

for name, df in sources.items():
    print(f"{name:20s}: {df.count():>6,} lignes")

26/04/27 22:51:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Google Searches     : 55,827 lignes


YouTube             : 14,076 lignes


Chrome              :    338 lignes


Spotify             :  5,926 lignes


Netflix             :  4,288 lignes


In [3]:
# ── 2. ANALYSE FRÉQUENTIELLE PAR SOURCE ───────────────────────────────────────
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# Stopwords : anglais + français + bruit technique
STOPWORDS_EXTRA = [
    # Français
    "les", "des", "une", "sur", "avec", "dans", "qui", "que", "par", "plus",
    "tout", "bien", "comme", "mais", "mon", "ton", "son", "nos", "mes",
    "faire", "comment", "plus", "aussi", "encore", "très",
    # Anglais générique
    "the", "and", "for", "with", "you", "your", "this", "that", "from",
    "are", "was", "not", "its", "but", "all", "new", "best", "how",
    # Bruit technique (URLs, tracking, ads)
    "https", "http", "www", "com", "org", "net", "html", "php", "utm",
    "amp", "utm_source", "befr", "dgoogle", "watch", "video", "clip",
    "official", "officiel", "youtube", "shorts",
]

STOP_ALL = StopWordsRemover.loadDefaultStopWords("english") \
         + StopWordsRemover.loadDefaultStopWords("french") \
         + STOPWORDS_EXTRA

results = {}

for source_name, df in sources.items():
    clean = df.filter(
        F.col("text").isNotNull() &
        (F.length(F.col("text")) > 2) &
        (~F.col("text").rlike(r'^https?://'))   # exclure les URLs brutes
    )

    tokenized = Tokenizer(inputCol="text", outputCol="words").transform(clean)
    filtered  = StopWordsRemover(
        inputCol="words", outputCol="tokens", stopWords=STOP_ALL
    ).transform(tokenized)

    freq = (
        filtered
        .select(F.explode("tokens").alias("word"))
        .filter(F.length("word") > 2)
        .groupBy("word")
        .count()
        .orderBy(F.desc("count"))
    )

    results[source_name] = freq
    print(f"✓ {source_name}")

print("\nAnalyse terminée.")

✓ Google Searches
✓ YouTube
✓ Chrome
✓ Spotify
✓ Netflix

Analyse terminée.


In [4]:
# ── 3. TOP 25 MOTS PAR SOURCE ─────────────────────────────────────────────────
TOP_N = 25

for source_name, freq_df in results.items():
    print(f"\n{'='*50}")
    print(f"  {source_name}")
    print(f"{'='*50}")
    freq_df.show(TOP_N, truncate=False)


  Google Searches


+----------+-----+
|word      |count|
+----------+-----+
|belgique  |309  |
|one       |268  |
|fifa      |261  |
|google    |240  |
|streaming |229  |
|piece     |208  |
|minecraft |204  |
|%c3%a0    |189  |
|prix      |178  |
|hannut    |159  |
|fortnite  |159  |
|synonyme  |154  |
|musique   |150  |
|mac       |144  |
|discord   |144  |
|pdf       |139  |
|film      |137  |
|carte     |137  |
|download  |137  |
|traduction|136  |
|pro       |135  |
|mp3       |132  |
|font      |131  |
|france    |129  |
|club      |126  |
+----------+-----+
only showing top 25 rows


  YouTube


+---------+-----+
|word     |count|
+---------+-----+
|16x9     |615  |
|inh      |523  |
|(clip    |284  |
|officiel)|284  |
|mix      |263  |
|live     |218  |
|(official|215  |
|vid      |213  |
|video)   |206  |
|1920x1080|192  |
|15s      |191  |
|music    |188  |
|house    |178  |
|2024     |177  |
|2025     |175  |
|(ft      |165  |
|ft.      |164  |
|squeezie |159  |
|one      |142  |
|#shorts  |130  |
|chrome   |128  |
|set      |127  |
|film     |126  |
|cards    |115  |
|j'ai     |114  |
+---------+-----+
only showing top 25 rows


  Chrome
+-----------------------+-----+
|word                   |count|
+-----------------------+-----+
|google                 |45   |
|recherche              |44   |
|arnaudleroy20@gmail.com|24   |
|gmail                  |24   |
|online                 |22   |
|nozgap                 |18   |
|photopea               |18   |
|editor                 |16   |
|photo                  |16   |
|stuttgart              |15   |
|streaming              |1

+---------+-----+
|word     |count|
+---------+-----+
|trio     |37   |
|lil      |20   |
|black    |17   |
|john     |16   |
|james    |16   |
|band     |15   |
|jack     |14   |
|big      |13   |
|orchestra|12   |
|michael  |12   |
|project  |12   |
|jay      |11   |
|david    |11   |
|jazz     |11   |
|king     |11   |
|music    |10   |
|michel   |10   |
|paul     |10   |
|daniel   |10   |
|sam      |9    |
|alex     |9    |
|mr.      |9    |
|young    |9    |
|ryan     |9    |
|jones    |9    |
+---------+-----+
only showing top 25 rows


  Netflix
+---------+-----+
|word     |count|
+---------+-----+
|naruto   |690  |
|shippuden|468  |
|hunter   |296  |
|(2011)   |148  |
|fairy    |147  |
|tail     |147  |
|adventure|117  |
|seven    |116  |
|jojo's   |109  |
|bizarre  |109  |
|sins     |91   |
|deadly   |91   |
|show     |90   |
|death    |85   |
|boruto   |80   |
|regular  |80   |
|horseman |77   |
|bojack   |77   |
|fullmetal|70   |
|alchemist|70   |
|baki     |65   |
|prison  

In [5]:
# ── 4. BIGRAMMES — Termes composés importants ─────────────────────────────────
# Les bigrammes capturent des concepts que les mots seuls manquent :
# "travis scott", "league of legends", "formula 1", "deep learning", etc.
from pyspark.ml.feature import NGram

bigram_results = {}

for source_name, df in sources.items():
    clean = df.filter(
        F.col("text").isNotNull() &
        (F.length(F.col("text")) > 2) &
        (~F.col("text").rlike(r'^https?://'))
    )

    tokenized = Tokenizer(inputCol="text", outputCol="words").transform(clean)
    filtered  = StopWordsRemover(
        inputCol="words", outputCol="tokens", stopWords=STOP_ALL
    ).transform(tokenized)

    bigrams = NGram(n=2, inputCol="tokens", outputCol="ngrams").transform(filtered)

    freq = (
        bigrams
        .select(F.explode("ngrams").alias("bigram"))
        .filter(F.length("bigram") > 5)
        .groupBy("bigram")
        .count()
        .orderBy(F.desc("count"))
    )

    bigram_results[source_name] = freq

print("Top bigrammes par source :\n")
for source_name, freq_df in bigram_results.items():
    print(f"── {source_name}")
    freq_df.show(15, truncate=False)
    print()

Top bigrammes par source :

── Google Searches


+---------------+-----+
|bigram         |count|
+---------------+-----+
|one piece      |194  |
|fifa 21        |148  |
|fifa 23        |59   |
|ralph lauren   |46   |
|streaming vf   |42   |
|travis scott   |42   |
|airpods pro    |39   |
|epic games     |39   |
|google flight  |37   |
|freeze corleone|37   |
|virtual dj     |36   |
|rocket league  |35   |
|stg gege       |33   |
|polo ralph     |33   |
|star wars      |32   |
+---------------+-----+
only showing top 15 rows


── YouTube


+--------------------+-----+
|bigram              |count|
+--------------------+-----+
|(clip officiel)     |280  |
|16x9 6s             |264  |
|vid 16x9            |195  |
|inh inh             |195  |
|choisissez chrome   |110  |
|- rediffusion       |110  |
|rediffusion squeezie|110  |
|inh cards           |106  |
|travis scott        |103  |
|inazuma eleven      |97   |
|music video)        |90   |
|(official video)    |86   |
|(official music     |85   |
|eleven -            |84   |
|one piece           |74   |
+--------------------+-----+
only showing top 15 rows


── Chrome
+-------------------------+-----+
|bigram                   |count|
+-------------------------+-----+
|- recherche              |41   |
|recherche google         |40   |
|- arnaudleroy20@gmail.com|24   |
|- gmail                  |24   |
|arnaudleroy20@gmail.com -|24   |
|online photo             |16   |
|photopea |               |16   |
|| online                 |16   |
|photo editor             |16   |
|noz

+---------------+-----+
|bigram         |count|
+---------------+-----+
|jazz trio      |3    |
|michel fugain  |2    |
|arsenal f.c.   |2    |
|miura jam      |2    |
|sound machine  |2    |
|mc faísca      |2    |
|coleman hawkins|2    |
|petite culotte |2    |
|jolem sanchez  |2    |
|red hot        |2    |
|red garland    |2    |
|oscar peterson |2    |
|& ryan         |2    |
|francis lai    |2    |
|los del        |2    |
+---------------+-----+
only showing top 15 rows


── Netflix
+-------------------+-----+
|bigram             |count|
+-------------------+-----+
|naruto shippuden   |468  |
|x hunter           |148  |
|hunter (2011)      |148  |
|hunter x           |148  |
|fairy tail         |147  |
|jojo's bizarre     |109  |
|bizarre adventure  |109  |
|deadly sins        |91   |
|seven deadly       |91   |
|regular show       |80   |
|bojack horseman    |77   |
|fullmetal alchemist|70   |
|rick morty         |61   |
|kusuo ψ            |56   |
|saiki kusuo        |56   |
+-

In [6]:
# ── 5. GAP ANALYSIS — Termes fréquents non couverts par CATEGORY_KEYWORDS ─────
# On identifie les mots qui apparaissent souvent mais ne sont dans aucune catégorie.

CATEGORY_KEYWORDS = {
    "Sport":          ["football","soccer","nba","match","goal","arsenal","fifa","ligue","rugby",
                       "tennis","basketball","sport","ucl","premier league","ufc","mma","boxing",
                       "gym","fitness","workout","calisthenics","training","nfl","olympics",
                       "swimming","cycling","padel","volleyball","champions league","ligue 1"],
    "Auto/Moto":      ["car","auto","voiture","porsche","ferrari","lamborghini","bmw","mercedes",
                       "audi","tesla","f1","formula 1","supercar","hypercar","drift","tuning",
                       "engine","motorsport","motorcycle","moto","yamaha","kawasaki","mclaren",
                       "bugatti","ducati","harley","grand prix","jdm","supra","amg"],
    "Musique":        ["music","song","artist","rap","album","track","beat","drill","trap",
                       "afrobeat","afropop","rnb","r&b","hip","hop","spotify","playlist",
                       "concert","festival","lyrics","producer","techno","house","electro",
                       "lo-fi","jazz","dj","remix","soundcloud","damso","tiakola","ninho",
                       "zamdane","gazo","niska","bezbar","travis scott","freeze corleone"],
    "Tech":           ["python","code","data","dev","javascript","api","ai","software","tech",
                       "developer","engineering","technology","artificial intelligence",
                       "machine learning","deep learning","nlp","llm","pytorch","tensorflow",
                       "backend","frontend","react","github","docker","kubernetes","cloud",
                       "aws","cybersecurity","linux","startup","chatgpt","openai","gpu","nvidia"],
    "Cinema/Series":  ["netflix","film","série","movie","episode","cinema","trailer","season",
                       "streaming","anime","manga","hbo","marvel","star wars","oscars",
                       "naruto","shippuden","fairy tail","jojo","baki","boruto","fullmetal",
                       "hunter","ghibli","rick morty","one piece"],
    "Gaming":         ["game","gaming","xbox","ps5","steam","minecraft","fortnite","esport",
                       "gamer","nintendo","switch","twitch","discord","multiplayer","rpg",
                       "fps","roblox","league of legends","valorant","warzone","gta",
                       "elden ring","zelda","playstation"],
    "Actu/Societe":   ["news","actu","monde","france","afrique","africa","belgique","politique",
                       "environment","ecology","climate","space","nasa","spacex","economy",
                       "finance","crypto","bitcoin","blockchain","stock market","history",
                       "philosophy"],
    "Shopping":       ["amazon","shop","brand","adidas","nike","fashion","streetwear","sneakers",
                       "yeezy","jordan","clothes","outfit","ecommerce","unboxing","skincare",
                       "watches","apple","iphone","samsung","gadget","dior","louis vuitton"],
    "Photo/Crea":     ["photo","photography","design","creative","art","visual","camera",
                       "graphic","illustration","photoshop","lightroom","editing",
                       "content creation","tiktok","reels","architecture","digital art",
                       "ui/ux","3d modeling","blender","canva"],
}

all_kw = {kw for kws in CATEGORY_KEYWORDS.values() for kw in kws}

# Union de toutes les sources pour la vue globale
from functools import reduce
all_sources = reduce(lambda a, b: a.union(b), sources.values())
clean_all = all_sources.filter(
    F.col("text").isNotNull() &
    (F.length(F.col("text")) > 2) &
    (~F.col("text").rlike(r'^https?://'))
)
tokenized_all = Tokenizer(inputCol="text", outputCol="words").transform(clean_all)
filtered_all  = StopWordsRemover(inputCol="words", outputCol="tokens", stopWords=STOP_ALL).transform(tokenized_all)

global_freq = (
    filtered_all
    .select(F.explode("tokens").alias("word"))
    .filter(F.length("word") > 2)
    .groupBy("word").count()
    .orderBy(F.desc("count"))
)

top_words = [row["word"] for row in global_freq.limit(200).collect()]
uncovered = [w for w in top_words if w not in all_kw]

print("Top 40 mots fréquents NON couverts par CATEGORY_KEYWORDS :")
print("(candidats à ajouter dans une catégorie)\n")
for w in uncovered[:40]:
    print(f"  {w}")

Top 40 mots fréquents NON couverts par CATEGORY_KEYWORDS :
(candidats à ajouter dans une catégorie)

  16x9
  inh
  one
  piece
  google
  (clip
  officiel)
  mix
  pro
  live
  2024
  (official
  vid
  club
  prix
  squeezie
  video)
  2025
  1920x1080
  15s
  %c3%a0
  mac
  saison
  musique
  black
  download
  ft.
  (ft
  hannut
  jbl
  show
  travis
  synonyme
  jeu
  scott
  fairy
  tail
  (2011)
  carte
  chrome


In [7]:
spark.stop()
print("Spark session fermée.")

Spark session fermée.
